# 07 – Portfolio Risk Report

Join positions to the chain, compute IV and Greeks, aggregate risk and run spot/vol scenarios.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import matplotlib.pyplot as plt
from scripts.portfolio_risk_report import (
    _enrich_chain, _build_position_greeks, _aggregate_risk, _scenario_pnl
)


In [ ]:
chain = pd.read_csv("../data/sample_chain.csv")
positions = pd.read_csv("../data/sample_positions.csv")
print("Positions:")
print(positions.to_string(index=False))


## Enrich chain with IV and Greeks

In [ ]:
chain_enriched = _enrich_chain(chain, r=0.05, q=0.0)
print(chain_enriched[["underlying","expiry","cp","K","T","mid","iv","delta","gamma","vega","theta"]].head(8).round(4).to_string(index=False))


## Build position-level Greeks

In [ ]:
pos_greeks = _build_position_greeks(positions, chain_enriched)
display_cols = ["underlying","expiry","cp","K","qty","iv","delta","dollar_delta","dollar_vega","dollar_theta"]
print(pos_greeks[display_cols].round(2).to_string(index=False))


## Aggregate risk by underlying and expiry

In [ ]:
agg = _aggregate_risk(pos_greeks)

print("\nRisk by underlying:")
print(agg["by_underlying"].round(2).to_string(index=False))

print("\nRisk by expiry:")
print(agg["by_expiry"].round(2).to_string(index=False))


## Scenario PnL

First-order (delta + vega) PnL across a grid of spot and vol shocks.

In [ ]:
spot_shocks = [-0.10, -0.05, -0.02, 0.0, 0.02, 0.05, 0.10]
vol_shocks  = [-0.05, -0.02, 0.0, 0.02, 0.05]
scen = _scenario_pnl(pos_greeks, spot_shocks, vol_shocks)

pivot = scen.pivot(index="spot_shock_pct", columns="vol_shock_pts", values="pnl")
print(pivot.round(0).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], marker="o", label=f"vol shock {col:+.0f}pp")
ax.axhline(0, linestyle="--", color="grey", alpha=0.5)
ax.set_xlabel("Spot shock (%)")
ax.set_ylabel("PnL ($)")
ax.set_title("Scenario PnL – Spot vs Vol shocks")
ax.legend(title="Vol shock (pp)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
